# COVID-19 Policy & Partisan Alignment

**CAIPH Datathon 2026**

Can COVID-19 policy choices predict which party a state's governor belongs to? This notebook trains machine learning models on 14 Oxford NPI policy features from 50 US states across 2020–2022 and evaluates how well policy alone classifies governor party over time.

| | |
|---|---|
| **Dataset** | COVID-19 Data Hub — 77,897 state-day rows, 50 US states, 1,096 days |
| **Features** | 14 Oxford NPI ordinal policy measures |
| **Targets** | Governor party (binary) · Party of Partisan Lean (binary) · Partisan lean (continuous) |
| **Split** | 70/30 by state — 35 train, 15 val (random seed=42) |
| **Models** | Logistic Regression (C=0.1) · Random Forest · Gradient Boosting · Transformer |

**Best full-sequence Transformer F1: 0.870** (13/15 held-out states correct)

In [1]:
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, r2_score, classification_report
from sklearn.preprocessing import LabelEncoder
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Paths relative to notebooks/
DATA_DIR    = os.path.join("..", "data")
OUT_DIR     = os.path.join("..", "outputs")
MAIN_CSV    = os.path.join(DATA_DIR, "covid19dh_usa_party_joint.csv")
PARTY_CSV   = os.path.join(DATA_DIR, "state_party_affiliation_2020_2021.csv")
LEAN_CSV    = os.path.join(DATA_DIR, "pol_lean.csv")

FEATURES = [
    "school_closing", "workplace_closing", "cancel_events",
    "gatherings_restrictions", "transport_closing", "stay_home_restrictions",
    "internal_movement_restrictions", "international_movement_restrictions",
    "information_campaigns", "testing_policy", "contact_tracing",
    "facial_coverings", "vaccination_policy", "elderly_people_protection",
]
STATE_COL = "State_x"

# Color scheme: darker = coarser time scale (quarterly darkest)
COLORS = {
    "lr_quarterly": "#1a3a6b",   # darkest blue
    "lr_monthly":   "#2e6db4",
    "lr_7d":        "#89b4e8",   # lightest blue
    "tf_quarterly": "#7a2800",   # darkest orange
    "tf_monthly":   "#c94a00",
    "tf_7d":        "#f4a46a",   # lightest orange
}
LINE_WIDTHS = {"quarterly": 3, "monthly": 2, "7d": 1}

# Canonical split
def make_split(state_arr, ratio=0.7, seed=42):
    rng  = np.random.RandomState(seed)
    perm = rng.permutation(len(state_arr))
    split = int(ratio * len(state_arr))
    return set(state_arr[perm[:split]]), set(state_arr[perm[split:]])

print("Imports done.")

Imports done.


## 1. Governor Party Map

In [2]:
STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY",
}
STATE_CENTROIDS = {
    "Alabama": (32.8,-86.8), "Alaska": (64.2,-153.0), "Arizona": (34.3,-111.1),
    "Arkansas": (34.8,-92.2), "California": (37.2,-119.5), "Colorado": (39.0,-105.5),
    "Connecticut": (41.6,-72.7), "Delaware": (39.0,-75.5), "Florida": (28.5,-82.5),
    "Georgia": (32.7,-83.4), "Hawaii": (20.9,-157.0), "Idaho": (44.4,-114.6),
    "Illinois": (40.0,-89.2), "Indiana": (39.9,-86.3), "Iowa": (42.1,-93.5),
    "Kansas": (38.5,-98.4), "Kentucky": (37.5,-85.3), "Louisiana": (31.2,-91.8),
    "Maine": (45.4,-69.2), "Maryland": (39.0,-76.8), "Massachusetts": (42.3,-71.8),
    "Michigan": (44.3,-85.4), "Minnesota": (46.4,-93.1), "Mississippi": (32.7,-89.7),
    "Missouri": (38.4,-92.5), "Montana": (47.0,-110.0), "Nebraska": (41.5,-99.9),
    "Nevada": (39.3,-116.6), "New Hampshire": (43.7,-71.6), "New Jersey": (40.1,-74.5),
    "New Mexico": (34.5,-106.1), "New York": (42.9,-75.5), "North Carolina": (35.5,-79.4),
    "North Dakota": (47.5,-100.5), "Ohio": (40.4,-82.8), "Oklahoma": (35.6,-97.5),
    "Oregon": (43.9,-120.6), "Pennsylvania": (40.9,-77.8), "Rhode Island": (41.7,-71.5),
    "South Carolina": (33.9,-80.9), "South Dakota": (44.4,-100.2), "Tennessee": (35.9,-86.4),
    "Texas": (31.5,-99.3), "Utah": (39.3,-111.1), "Vermont": (44.1,-72.7),
    "Virginia": (37.5,-79.4), "Washington": (47.4,-120.5), "West Virginia": (38.6,-80.6),
    "Wisconsin": (44.6,-89.9), "Wyoming": (43.0,-107.6),
}

party_df = pd.read_csv(PARTY_CSV)
lean_df  = pd.read_csv(LEAN_CSV)[["State", "Partisan Lean", "Party of Partisan Lean"]]
map_df   = party_df.merge(lean_df, on="State", how="inner")
map_df["abbrev"] = map_df["State"].map(STATE_ABBREV)
map_df["z"]      = map_df["Party"].map({"Republican": 1, "Democratic": 0})
map_df["lat"]    = map_df["State"].map(lambda s: STATE_CENTROIDS[s][0])
map_df["lon"]    = map_df["State"].map(lambda s: STATE_CENTROIDS[s][1])
lean_sign        = map_df["Partisan Lean"].apply(lambda v: f"+{v}" if v > 0 else str(v))
map_df["hover"]  = (map_df["State"] + "<br>Governor: " + map_df["Party"] +
                    "<br>Lean: " + lean_sign + " (" + map_df["Party of Partisan Lean"] + ")")

fig_map = go.Figure(data=[
    go.Choropleth(
        locations=map_df["abbrev"], z=map_df["z"],
        locationmode="USA-states",
        colorscale=[[0, "#1a5fad"], [1, "#c01a2a"]],
        zmin=0, zmax=1, showscale=False,
        hovertext=map_df["hover"], hoverinfo="text",
    ),
    go.Scattergeo(
        lat=map_df["lat"], lon=map_df["lon"],
        text=lean_sign, mode="text",
        textfont=dict(color="white", size=9, family="Arial Black"),
        hoverinfo="skip",
    ),
])
fig_map.update_layout(
    title=dict(text="US Governor Party Affiliation & Partisan Lean (2020–2021)", x=0.5),
    geo=dict(scope="usa", showlakes=False),
    margin=dict(l=0, r=0, t=50, b=0), height=480,
)
fig_map.show()

## 2. Feature Importance

Which policy features best distinguish Republican from Democratic governors? Evaluated on a row-level 80/20 train/test split.

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(MAIN_CSV, parse_dates=["date"])
df = df.dropna(subset=FEATURES).copy()
le = LabelEncoder()
df["party_enc"] = le.fit_transform(df["Party"])
le_ppl = LabelEncoder()
df["ppl_enc"] = le_ppl.fit_transform(df["Party of Partisan Lean"])

X = df[FEATURES].values
y = df["party_enc"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, C=0.1),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, random_state=42),
}

importances = {}
print(f"{'Model':<25} {'F1':>8}")
print("-" * 35)
for name, m in clf_models.items():
    m.fit(X_tr, y_tr)
    f1 = f1_score(y_te, m.predict(X_te), average="weighted")
    importances[name] = np.abs(m.coef_[0]) if hasattr(m, "coef_") else m.feature_importances_
    print(f"{name:<25} {f1:>8.4f}")

Model                           F1
-----------------------------------


Logistic Regression         0.6501


Random Forest               0.8989


Gradient Boosting           0.8148


In [4]:
fig_imp = make_subplots(rows=1, cols=3, subplot_titles=list(importances.keys()))
for col, (name, imp) in enumerate(importances.items(), 1):
    order = np.argsort(imp)
    fig_imp.add_trace(
        go.Bar(x=imp[order], y=[FEATURES[i] for i in order],
               orientation="h", name=name,
               marker_color=["#1a5fad", "#c01a2a", "#2e8b57"][col - 1]),
        row=1, col=col
    )
fig_imp.update_layout(
    title="Feature Importance — Party Classification",
    height=500, showlegend=False
)
fig_imp.show()

## 3. Temporal Party Predictability

Party F1 evaluated over time using two models:
- **LR probe** (blue): Logistic Regression retrained on each window's training data
- **Transformer** (orange): Frozen best-checkpoint model, applied to each window without retraining

Darker shades = shorter time scale (7-day darkest, quarterly lightest).

In [5]:
# Load data and build per-state sequences
states     = np.array(sorted(df[STATE_COL].unique()))
date_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")

sequences, party_labels, lean_labels, ppl_labels = [], [], [], []
for state in states:
    sdf = df[df[STATE_COL] == state].set_index("date").reindex(date_range)
    sequences.append(sdf[FEATURES].fillna(0).values.astype(np.float32))
    party_labels.append(df[df[STATE_COL] == state]["party_enc"].iloc[0])
    lean_labels.append(df[df[STATE_COL] == state]["Partisan Lean"].iloc[0])
    ppl_labels.append(df[df[STATE_COL] == state]["ppl_enc"].iloc[0])

y_party_seq = np.array(party_labels)
state_names = states

# Canonical 70/30 split
n_states = len(state_names)
rng      = np.random.RandomState(42)
perm     = rng.permutation(n_states)
split    = int(0.7 * n_states)
tr_idx, va_idx = perm[:split], perm[split:]
train_states   = set(state_names[tr_idx])
test_states    = set(state_names[va_idx])
val_state_list = list(state_names[va_idx])

print(f"Train: {len(tr_idx)} states | Val: {len(va_idx)} states")
print(f"Val states: {val_state_list}")

Train: 35 states | Val: 15 states
Val states: [np.str_('Arizona'), np.str_('Utah'), np.str_('Oklahoma'), np.str_('Mississippi'), np.str_('South Dakota'), np.str_('Hawaii'), np.str_('Minnesota'), np.str_('Maine'), np.str_('Wyoming'), np.str_('Massachusetts'), np.str_('Delaware'), np.str_('Texas'), np.str_('Iowa'), np.str_('New Hampshire'), np.str_('Rhode Island')]


In [6]:
# Load Transformer checkpoints
class PartyTransformer(nn.Module):
    def __init__(self, n_features=14, d_model=32, nhead=4, num_layers=2, n_out=1, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True,
                                               dim_feedforward=128, dropout=dropout)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.classifier  = nn.Sequential(nn.Linear(d_model, 16), nn.ReLU(), nn.Linear(16, n_out))
    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x).squeeze(-1)

device = torch.device("cpu")
model_party = PartyTransformer(n_features=14, n_out=1).to(device)
model_party.load_state_dict(torch.load(os.path.join(OUT_DIR, "transformer_party_best.pt"), map_location=device))
model_party.eval()

model_reg = PartyTransformer(n_features=14, n_out=1).to(device)
model_reg.load_state_dict(torch.load(os.path.join(OUT_DIR, "transformer_lean_reg_best.pt"), map_location=device))
model_reg.eval()

probe_party = LogisticRegression(max_iter=1000, random_state=42, C=0.1)
probe_ppl   = LogisticRegression(max_iter=1000, random_state=42, C=0.1)
probe_reg   = LinearRegression()

def build_val_seqs(window_dates):
    seqs, y_p, y_l, y_ppl = [], [], [], []
    for state in val_state_list:
        sdf = df[df[STATE_COL] == state].set_index("date").reindex(window_dates)
        seqs.append(sdf[FEATURES].fillna(0).values.astype(np.float32))
        row = df[df[STATE_COL] == state].iloc[0]
        y_p.append(row["party_enc"])
        y_l.append(row["Partisan Lean"])
        y_ppl.append(row["ppl_enc"])
    return np.stack(seqs), np.array(y_p), np.array(y_l, dtype=np.float32), np.array(y_ppl)

def score_window(window_dates, tr_wdf, te_wdf):
    rec = {}
    if len(tr_wdf["party_enc"].unique()) >= 2:
        probe_party.fit(tr_wdf[FEATURES], tr_wdf["party_enc"])
        rec["lr_f1"] = f1_score(te_wdf["party_enc"],
                                 probe_party.predict(te_wdf[FEATURES]),
                                 average="weighted", zero_division=0)
    if len(tr_wdf["ppl_enc"].unique()) >= 2:
        probe_ppl.fit(tr_wdf[FEATURES], tr_wdf["ppl_enc"])
        rec["lr_ppl_f1"] = f1_score(te_wdf["ppl_enc"],
                                     probe_ppl.predict(te_wdf[FEATURES]),
                                     average="weighted", zero_division=0)
    seqs, y_p, _, y_ppl = build_val_seqs(window_dates)
    X_te = torch.tensor(seqs).to(device)
    with torch.no_grad():
        preds = (model_party(X_te).cpu().numpy() > 0).astype(int)
        rec["tf_f1"]     = f1_score(y_p,   preds, average="weighted", zero_division=0)
        rec["tf_ppl_f1"] = f1_score(y_ppl, preds, average="weighted", zero_division=0)
    return rec

print("Models loaded.")

Models loaded.


In [7]:
# Quarterly windows (anchored to 2020-03-01, 91-day windows)
print("Running quarterly...")
anchor   = pd.Timestamp("2020-03-01")
q_starts = pd.date_range(anchor, df["date"].max(), freq="91D")
quarterly = []
for q_start in q_starts:
    q_end   = q_start + pd.Timedelta(days=90)
    q_dates = pd.date_range(q_start, q_end, freq="D")
    qdf     = df[df["date"].isin(q_dates)].dropna(subset=FEATURES)
    tr_qdf  = qdf[qdf[STATE_COL].isin(train_states)]
    te_qdf  = qdf[qdf[STATE_COL].isin(test_states)]
    if len(te_qdf) == 0: continue
    rec = score_window(q_dates, tr_qdf, te_qdf)
    rec["date"] = q_start.strftime("%Y-%m-%d"); rec["date_dt"] = q_start.to_pydatetime()
    quarterly.append(rec)
quarterly_df = pd.DataFrame(quarterly)

# Monthly windows
print("Running monthly...")
months, monthly = pd.period_range(df["date"].min(), df["date"].max(), freq="M"), []
for month in months:
    mdf = df[df["date"].dt.to_period("M") == month].dropna(subset=FEATURES)
    tr_mdf = mdf[mdf[STATE_COL].isin(train_states)]
    te_mdf = mdf[mdf[STATE_COL].isin(test_states)]
    if len(te_mdf) == 0: continue
    m_dates = pd.date_range(str(month.start_time.date()), str(month.end_time.date()), freq="D")
    rec = score_window(m_dates, tr_mdf, te_mdf)
    rec["date"] = str(month); rec["date_dt"] = month.start_time.to_pydatetime()
    monthly.append(rec)
monthly_df = pd.DataFrame(monthly)

print(f"Done. Quarterly: {len(quarterly_df)}, Monthly: {len(monthly_df)}")

Running quarterly...


Running monthly...


Done. Quarterly: 12, Monthly: 36


### 3a. LR Probe — All Time Scales

In [8]:
fig_lr = go.Figure()
for tdf, scale, scale_key in [
    (quarterly_df, "Quarterly", "quarterly"),
    (monthly_df,   "Monthly",   "monthly"),
]:
    if "lr_f1" in tdf.columns:
        fig_lr.add_trace(go.Scatter(
            x=tdf["date_dt"], y=tdf["lr_f1"], mode="lines", name=scale,
            customdata=tdf["date"],
            hovertemplate="%{customdata}: %{y:.3f}<extra>" + scale + "</extra>",
            line=dict(color=COLORS[f"lr_{scale_key}"], width=LINE_WIDTHS[scale_key]),
        ))
fig_lr.update_layout(
    title="Party Predictability — LR Probe (Quarterly & Monthly)",
    xaxis_title="Date", yaxis_title="Weighted F1",
    yaxis=dict(range=[0, 1]), hovermode="x unified", height=480,
)
fig_lr.show()

### 3b. Transformer — All Time Scales

In [9]:
fig_tf = go.Figure()
for tdf, scale, scale_key in [
    (quarterly_df, "Quarterly", "quarterly"),
    (monthly_df,   "Monthly",   "monthly"),
]:
    if "tf_f1" in tdf.columns:
        fig_tf.add_trace(go.Scatter(
            x=tdf["date_dt"], y=tdf["tf_f1"], mode="lines", name=scale,
            customdata=tdf["date"],
            hovertemplate="%{customdata}: %{y:.3f}<extra>" + scale + "</extra>",
            line=dict(color=COLORS[f"tf_{scale_key}"], width=LINE_WIDTHS[scale_key]),
        ))
fig_tf.update_layout(
    title="Party Predictability — Transformer (Quarterly & Monthly)",
    xaxis_title="Date", yaxis_title="Weighted F1",
    yaxis=dict(range=[0, 1]), hovermode="x unified", height=480,
)
fig_tf.show()

## 4. Temporal "Party of Partisan Lean" Predictability

F1 for predicting the **Party of Partisan Lean** (Democrat/Republican based on Cook Political Report lean score) rather than governor party. Uses the same LR probe and frozen Transformer.

In [10]:
fig_lr_ppl = go.Figure()
for tdf, scale, scale_key in [
    (quarterly_df, "Quarterly", "quarterly"),
    (monthly_df,   "Monthly",   "monthly"),
]:
    if "lr_ppl_f1" in tdf.columns:
        fig_lr_ppl.add_trace(go.Scatter(
            x=tdf["date_dt"], y=tdf["lr_ppl_f1"], mode="lines", name=scale,
            customdata=tdf["date"],
            hovertemplate="%{customdata}: %{y:.3f}<extra>" + scale + "</extra>",
            line=dict(color=COLORS[f"lr_{scale_key}"], width=LINE_WIDTHS[scale_key]),
        ))
fig_lr_ppl.update_layout(
    title="Party of Partisan Lean Predictability — LR Probe (Quarterly & Monthly)",
    xaxis_title="Date", yaxis_title="Weighted F1",
    yaxis=dict(range=[0, 1]), hovermode="x unified", height=480,
)
fig_lr_ppl.show()

In [11]:
fig_tf_ppl = go.Figure()
for tdf, scale, scale_key in [
    (quarterly_df, "Quarterly", "quarterly"),
    (monthly_df,   "Monthly",   "monthly"),
]:
    if "tf_ppl_f1" in tdf.columns:
        fig_tf_ppl.add_trace(go.Scatter(
            x=tdf["date_dt"], y=tdf["tf_ppl_f1"], mode="lines", name=scale,
            customdata=tdf["date"],
            hovertemplate="%{customdata}: %{y:.3f}<extra>" + scale + "</extra>",
            line=dict(color=COLORS[f"tf_{scale_key}"], width=LINE_WIDTHS[scale_key]),
        ))
fig_tf_ppl.update_layout(
    title="Party of Partisan Lean Predictability — Transformer (Quarterly & Monthly)",
    xaxis_title="Date", yaxis_title="Weighted F1",
    yaxis=dict(range=[0, 1]), hovermode="x unified", height=480,
)
fig_tf_ppl.show()

In [12]:
fig_combined = go.Figure()
fig_combined.add_trace(go.Scatter(
    x=quarterly_df["date_dt"], y=quarterly_df["lr_f1"], mode="lines",
    name="LR — Party", customdata=quarterly_df["date"],
    hovertemplate="%{customdata}: %{y:.3f}<extra>LR — Party</extra>",
    line=dict(color="#1a3a6b", width=3, dash="solid"),
))
fig_combined.add_trace(go.Scatter(
    x=quarterly_df["date_dt"], y=quarterly_df["tf_f1"], mode="lines",
    name="TF — Party", customdata=quarterly_df["date"],
    hovertemplate="%{customdata}: %{y:.3f}<extra>TF — Party</extra>",
    line=dict(color="#7a2800", width=3, dash="solid"),
))
fig_combined.add_trace(go.Scatter(
    x=quarterly_df["date_dt"], y=quarterly_df["lr_ppl_f1"], mode="lines",
    name="LR — PPL", customdata=quarterly_df["date"],
    hovertemplate="%{customdata}: %{y:.3f}<extra>LR — PPL</extra>",
    line=dict(color="#1a3a6b", width=3, dash="dash"),
))
fig_combined.add_trace(go.Scatter(
    x=quarterly_df["date_dt"], y=quarterly_df["tf_ppl_f1"], mode="lines",
    name="TF — PPL", customdata=quarterly_df["date"],
    hovertemplate="%{customdata}: %{y:.3f}<extra>TF — PPL</extra>",
    line=dict(color="#7a2800", width=3, dash="dash"),
))
fig_combined.update_layout(
    title="Quarterly: Governor Party vs Party of Partisan Lean — LR & Transformer",
    xaxis_title="Date", yaxis_title="Weighted F1",
    yaxis=dict(range=[0, 1]), hovermode="x unified", height=480,
)
fig_combined.show()

## 5. Combined Quarterly: Party vs Party of Partisan Lean

All four model–target combinations on the same plot. Solid = Governor Party, dashed = Party of Partisan Lean. Blue = LR probe, orange = Transformer.

## 6. Summary — Old Quarterly Notebooks (LR Probe, Q1–Q8)

Results from 8 quarterly standalone notebooks, each trained on a 3-month COVID window starting March 2020. First STEP 4 = Governor Party, second STEP 4 = Party of Partisan Lean.

In [13]:
df_summary = pd.DataFrame({
    "Quarter": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8"],
    "Period":  [
        "2020-03 – 2020-06", "2020-06 – 2020-09", "2020-09 – 2020-12",
        "2020-12 – 2021-03", "2021-03 – 2021-06", "2021-06 – 2021-09",
        "2021-09 – 2021-12", "2021-12 – 2022-03",
    ],
    "Party F1 (wtd)": [0.60, 0.73, 0.65, 0.45, 0.60, 0.56, 0.53, 0.66],
    "Party AUC":      [0.679, 0.804, 0.821, 0.589, 0.768, 0.411, 0.527, 0.536],
    "PPL F1 (wtd)":   [0.54, 0.72, 0.72, 0.80, 0.60, 0.63, 0.66, 0.49],
    "PPL AUC":        [0.630, 0.685, 0.796, 0.796, 0.685, 0.722, 0.741, 0.722],
})

df_summary.style \
    .format({"Party F1 (wtd)": "{:.2f}", "Party AUC": "{:.3f}",
             "PPL F1 (wtd)": "{:.2f}", "PPL AUC": "{:.3f}"}) \
    .background_gradient(subset=["Party F1 (wtd)", "PPL F1 (wtd)"], cmap="Blues", vmin=0.3, vmax=0.9) \
    .background_gradient(subset=["Party AUC", "PPL AUC"], cmap="Oranges", vmin=0.3, vmax=0.9) \
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]) \
    .hide(axis="index")

ImportError: `Import matplotlib` failed. Styler.background_gradient requires matplotlib. Use pip or conda to install the matplotlib package.

In [14]:
fig_q = go.Figure()
fig_q.add_trace(go.Scatter(
    x=df_summary["Quarter"], y=df_summary["Party F1 (wtd)"], mode="lines+markers",
    name="Party F1", line=dict(color="#1a3a6b", width=3),
    marker=dict(size=8),
))
fig_q.add_trace(go.Scatter(
    x=df_summary["Quarter"], y=df_summary["PPL F1 (wtd)"], mode="lines+markers",
    name="PPL F1", line=dict(color="#7a2800", width=3, dash="dash"),
    marker=dict(size=8),
))
fig_q.update_layout(
    title="LR Probe F1 by Quarter — Governor Party vs Party of Partisan Lean",
    xaxis_title="Quarter", yaxis_title="Weighted F1",
    yaxis=dict(range=[0, 1]), hovermode="x unified", height=420,
)
fig_q.show()